In [2]:
# Testing Real-Data Verification for lcmv_stats

import numpy as np
import pandas as pd
from pathlib import Path
import lcmv_stats as ls

# ───────────── CONFIGURATION ────────────
PROJECT_BASE = Path("/mnt/movement/users/jaizor/xtra")
LCMV_ROOT = PROJECT_BASE / "derivatives/lcmv"
EVENTS_CSV = PROJECT_BASE / "data/csv/bima_events.csv"
OUTPUT_DIR = Path("./results/real_data_test")  # Relative to notebook
ALPHA_THRESHOLD = 0.01  # Toggle between 0.05 and 0.01

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Data Verification for lcmv_stats (Refactored - No Plots)...")

# ───────────── 1. ATLAS & UTILS CHECK ─────────────
print("\n1️⃣ Testing Atlas & Utils...")
try:
    labels = ls.get_cimt_labels()
    print(f"   ✅ Loaded {len(labels)} CIMT ROIs")

    motor_indices = ls.get_motor_network_indices()
    print(f"   ✅ Motor Network has {len(motor_indices)} ROIs")

    test_id = ls.map_subject_to_subj("Sbj001")
    print(f"   ✅ Mapping 'Sbj001' -> '{test_id}'")
except Exception as e:
    print(f"   ❌ Atlas/Utils Error: {e}")
    raise

# ───────────── 2. EPOCHING (Event-Locked) ─────────────
print("\n2️⃣ Testing Event-Locked Epoching...")
events_df = pd.read_csv(EVENTS_CSV)
if 'notes' in events_df.columns:
    events_df = events_df[events_df['notes'] == 'good']

subject_ids = [ls.map_subject_to_subj(name) for name in events_df['subject'].unique()]

if not subject_ids:
    raise ValueError("No subjects found in events CSV.")

test_subj = subject_ids[0]
subj_events = events_df[events_df['subject'].apply(lambda x: ls.map_subject_to_subj(x) == test_subj)]

in_ep, out_ep = ls.extract_event_epochs(
    subject_id=test_subj,
    lcmv_root=LCMV_ROOT,
    events_df=subj_events,
    pre_sec=5.0, post_sec=5.0,
    epoch_duration=2.0, overlap=0.8
)
print(f"   ✅ {test_subj}: {in_ep.shape[0]} in-phase epochs extracted")

# 📝 CONSOLE OUTPUT: Sample Epoch Structure
if in_ep.size > 0:
    sample_epochs = in_ep[:5, :3, :]
    df_epochs = pd.DataFrame(sample_epochs.reshape(-1, sample_epochs.shape[-1]))
    print("\n📋 SAMPLE EPOCH DATA (First 5 epochs, First 3 ROIs):")
    print(df_epochs.head(10).to_string())
    print(f"   ... ({len(df_epochs)} total rows)\n")

# ───────────── 3. CONNECTIVITY (WPLI) ─────────────
print("\n3️⃣ Testing WPLI Connectivity...")
sfreq = ls.get_subject_sfreq(test_subj, LCMV_ROOT)
in_conn, out_conn = ls.extract_wpli_features(
    epochs_in=in_ep, epochs_out=out_ep, band="low_beta", sfreq=sfreq
)

roi_names = []
if in_conn is not None:
    print(f"   ✅ WPLI Matrix shape: {in_conn.shape}")
    
    roi_info = ls._atlas.get_motor_network_metadata()
    roi_names = roi_info['roi_name'].tolist()
    
    df_in_conn = pd.DataFrame(in_conn, index=roi_names, columns=roi_names)
    df_out_conn = pd.DataFrame(out_conn, index=roi_names, columns=roi_names)
    
    # 📝 CONSOLE OUTPUT: Top 5x5 In-Phase Connectivity Submatrix
    print("\n📋 TOP 5x5 IN-PHASE CONNECTIVITY MATRIX:")
    print(df_in_conn.iloc[:5, :5].round(4).to_string())
    print("   ...\n")
else:
    print("   ️ Connectivity calculation returned None.")

# ──────────── 4. STATISTICS (Using Unified Batch Helper) ─────────────
print("\n4️ Testing Edge-Wise Statistics (Unified Workflow)...")

group_data = ls.prepare_group_comparison(
    events_df=events_df,
    lcmv_root=LCMV_ROOT,
    band="low_beta",
    is_phase_comparison=True
)

print(f"   ✅ Successfully processed {len(group_data['valid_subs'])} subjects.")

df_sig = pd.DataFrame()

if len(group_data['valid_subs']) >= 3:
    df_sig = ls.run_edgewise_permutation(
        in_data=group_data['data_a'], 
        out_data=group_data['data_b'], 
        n_permutations=1000
    )
    
    sig_count = len(df_sig[df_sig['p_val'] < ALPHA_THRESHOLD])
    print(f"   ✅ Found {sig_count} significant edges (p<{ALPHA_THRESHOLD})")

    if not df_sig.empty:
        # 📝 CONSOLE OUTPUT: Top 10 Significant Edges
        print("\n📋 TOP 10 SIGNIFICANT EDGES:")
        top_edges = df_sig.nlargest(10, 'abs_d')[
            ['roi1', 'roi2', 'cohens_d', 'p_val', 'mean_diff']
        ].round(4)
        print(top_edges.to_string(index=False))
        print("\n")
        
        # 💾 SAVE REPORT ONLY (No Plot)
        ls.generate_markdown_report(df_sig.assign(band="low_beta"), OUTPUT_DIR / "test_report.md")
        print("    Saved markdown report to disk")
else:
    print("   ⚠️ Not enough valid subjects for group statistics.")

# ───────────── 5. TARGETED GPDC + DIRECTED EFFECT MAP ────────────
print("\n5️⃣ Testing Targeted GPDC & Directed Effect Map...")
directed_map = pd.DataFrame()

if not df_sig.empty:
    top_sig_edges = df_sig.nlargest(5, 'abs_d')
    
    s_events = events_df[events_df['subject'].apply(lambda x: ls.map_subject_to_subj(x) == test_subj)]
    in_ep_test, out_ep_test = ls.extract_event_epochs(test_subj, LCMV_ROOT, s_events)
    
    sfreq = ls.get_subject_sfreq(test_subj, LCMV_ROOT)

    in_gpdc, out_gpdc, gpdc_roi_names = ls.extract_gpdc_features(
        epochs_in=in_ep_test,
        epochs_out=out_ep_test,
        sig_df=top_sig_edges,
        sfreq=sfreq,
        band_range=(13, 30)
    )

    if in_gpdc is not None:
        print(f"   ✅ GPDC computed for {len(gpdc_roi_names)} ROIs")
        
        # 📝 CONSOLE OUTPUT: GPDC Matrix Preview
        df_gpdc = pd.DataFrame(in_gpdc, index=gpdc_roi_names, columns=gpdc_roi_names)
        print("\n📋 GPDC MATRIX PREVIEW (First 5x5):")
        print(df_gpdc.iloc[:5, :5].round(4).to_string())
        print("   ...\n")

        # CREATE DIRECTED EFFECT MAP
        directed_map = ls.reporting.create_directed_effect_map(
            df_sig=df_sig,
            in_gpdc=in_gpdc,
            out_gpdc=out_gpdc,
            roi_names=gpdc_roi_names,
            alpha=ALPHA_THRESHOLD,
            condition_label="In-Phase"
        )
        
        if not directed_map.empty:
            # 📝 CONSOLE OUTPUT: Directed Effect Map
            print("\n🧭 DIRECTED EFFECT MAP (Significant WPLI + GPDC Direction):")
            print(directed_map[['edge', 'dominant_direction', 'directional_strength', 'cohens_d', 'p_val']].round(4).to_string(index=False))
            print(f"\n   Total directed edges: {len(directed_map)}\n")
        else:
            print("   ⚠️ No significant edges matched GPDC ROI subset for directed map.")
    else:
        print("   ⚠️ GPDC calculation returned None.")
else:
    print("   ⚠️ Skipping GPDC: No significant edges found.")

# ───────────── 6. SPECTROGRAM CLUSTERING ─────────────
print("\n6️⃣ Testing Spectrogram Clustering...")
spec_list = []
f_ref, t_ref = None, None
ROI_NAME = "STN-lh"
PRE_SEC = 5.0
BASELINE_DURATION = 2.0

try:
    roi_idx = ls._atlas.get_roi_index(ROI_NAME)
except Exception as e:
    print(f"   ⚠️ Could not find ROI index for {ROI_NAME}: {e}")
    roi_idx = None

if roi_idx is not None:
    for sid in subject_ids[:2]:
        s_events = events_df[events_df['subject'].apply(lambda x: ls.map_subject_to_subj(x) == sid)]
        i_ep, _ = ls.extract_event_epochs(sid, LCMV_ROOT, s_events)
        
        if i_ep.size > 0:
            avg_epoch = np.mean(i_ep, axis=0)
            signal_1d = avg_epoch[roi_idx, :]
            sfreq = ls.get_subject_sfreq(sid, LCMV_ROOT)
            
            f, t, Sxx_z = ls.compute_zscored_spectrogram(
                epoch=signal_1d,
                sfreq=sfreq,
                f_min=13.0,      # Restricted to Beta band
                f_max=30.0,      # Restricted to Beta band
                pre_sec=PRE_SEC,
                baseline_duration=BASELINE_DURATION
            )
            
            if f_ref is None:
                f_ref, t_ref = f, t
            spec_list.append(Sxx_z)

    if spec_list:
        spec_3d = np.stack(spec_list)
        T_obs, clusters, p_vals, _ = ls.run_cluster_spectrogram(
            spec_3d, 
            n_permutations=1000,
            threshold={'start': 0.1, 'step': 0.05}
        )
        sig_count = sum(1 for p in p_vals if p < 0.05)
        print(f"   ✅ Cluster test complete. Found {sig_count} significant clusters.")
        
        # 💾 SAVE NUMERICAL RESULTS ONLY (No Plot)
        np.savez_compressed(
            OUTPUT_DIR / "group_test_low_beta_spectral_stats.npz",
            f=f_ref, t=t_ref, avg_sxx=np.mean(spec_3d, axis=0),
            clusters=clusters, cluster_pvs=p_vals
        )
        print("   💾 Saved spectral numerical results to .npz")
    else:
        print("   ⚠️ No valid spectrograms generated.")
else:
    print("   ️ Skipping Spectrogram Clustering due to ROI error.")

print("\n🏁 Real-Data Verification Complete!")
print(f"   📁 Numerical outputs saved to: {OUTPUT_DIR.resolve()}")
print("   📋 All key data printed above for copy-paste inspection.")

Data Verification for lcmv_stats (Refactored - No Plots)...

1️⃣ Testing Atlas & Utils...
   ✅ Loaded 448 CIMT ROIs
   ✅ Motor Network has 80 ROIs
   ✅ Mapping 'Sbj001' -> 'sub-001'

2️⃣ Testing Event-Locked Epoching...
   ✅ sub-001: 16 in-phase epochs extracted

📋 SAMPLE EPOCH DATA (First 5 epochs, First 3 ROIs):
        0         1         2         3         4         5         6         7         8         9         10        11        12        13        14        15        16        17        18        19        20        21        22        23        24        25        26        27        28        29        30        31        32        33        34        35        36        37        38        39        40        41        42        43        44        45        46        47        48        49        50        51        52        53        54        55        56        57        58        59        60        61        62        63        64        65        66        67    

Maximum iterations reached. 0 of 1 converged
Maximum iterations reached. 0 of 1 converged
Maximum iterations reached. 0 of 1 converged
Maximum iterations reached. 0 of 1 converged
Maximum iterations reached. 0 of 1 converged
Maximum iterations reached. 0 of 1 converged


   ✅ GPDC computed for 9 ROIs

📋 GPDC MATRIX PREVIEW (First 5x5):
              CAU-tail-lh  CAU-tail-rh  L_2_ROI  L_a9-46v_ROI     M2L
CAU-tail-lh        0.1194       0.1056   0.1156        0.1199  0.1292
CAU-tail-rh        0.1020       0.1096   0.1084        0.0843  0.1184
L_2_ROI            0.1014       0.1067   0.1484        0.0800  0.1197
L_a9-46v_ROI       0.1508       0.1377   0.1179        0.2706  0.1157
M2L                0.0835       0.0888   0.0931        0.0706  0.0888
   ...


🧭 DIRECTED EFFECT MAP (Significant WPLI + GPDC Direction):
                edge     dominant_direction  directional_strength  cohens_d  p_val
CAU-tail-lh-R_6r_ROI R_6r_ROI → CAU-tail-lh                0.0414   -1.2216  0.005
L_2_ROI-L_a9-46v_ROI L_a9-46v_ROI → L_2_ROI                0.0379   -1.1004  0.007
     CAU-tail-rh-M4R      CAU-tail-rh → M4R                0.0240   -1.0972  0.001
          M3L-aGP-lh           aGP-lh → M3L                0.0133   -1.1869  0.004
          M2L-aGP-lh           